In [ ]:
import numpy as np
import glob
import os
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from skimage.io import imread
from skimage.transform import resize
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input   # Import VGG16 and preprocessing from TensorFlow Keras


# 1. Load images and ages from UTKFace dataset
data_path = '/Users/rongrongqian/Desktop/AI modules/regression/UTKface/part1/*.jpg' #change the path
image_files = glob.glob(data_path)

ages = []
images_resized = []
img_size = (224, 224)  # VGG16 requires 224x224 input

# Load and preprocess images
for file in image_files[:100]:  # Use the first 100 images for demonstration
    age = int(os.path.basename(file).split('_')[0])  # Age is the first part of filename
    img = imread(file)
    img_resized = resize(img, img_size, anti_aliasing=True)
    ages.append(age)
    images_resized.append(img_resized)

ages = np.array(ages)
images_resized = np.array(images_resized)  #Convert lists to NumPy arrays 
#for efficient numerical operations and compatibility with scikit-learn 
#and deep learning frameworks.

# 2. Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    images_resized, ages, test_size=0.2, random_state=42
)

# 3. Extract CNN features using pre-trained VGG16
def extract_vgg_features(images):
    """
    Extract VGG16 features from a list/array of images.
    Each image must be 224x224x3 and pixel values in [0, 1].
    """
    model = VGG16(weights='imagenet', include_top=False, pooling='avg')
    features = []
    for img in images:
        # Preprocessing: rescale to [0,255], then preprocess as VGG expects
        img_batch = np.expand_dims(img * 255.0, axis=0)  # shape (1,224,224,3)
        img_preprocessed = preprocess_input(img_batch)
        feature_vector = model.predict(img_preprocessed, verbose=0)
        features.append(feature_vector.flatten())
    return np.array(features)

print("Extracting VGG16 features for training set...")
X_train_vgg = extract_vgg_features(X_train)
print("Extracting VGG16 features for test set...")
X_test_vgg = extract_vgg_features(X_test)

# 4. Standardize VGG16 features
scaler = StandardScaler()
X_train_vgg_scaled = scaler.fit_transform(X_train_vgg)
X_test_vgg_scaled = scaler.transform(X_test_vgg)

# 5. OLS
vgg_model_ols = LinearRegression().fit(X_train_vgg_scaled, y_train)
vgg_pred_ols = vgg_model_ols.predict(X_test_vgg_scaled)
mae_ols = mean_absolute_error(y_test, vgg_pred_ols)
print("VGG16 Features OLS MAE:", mae_ols)

# 6. Ridge Regression with cross-validated alpha selection
# You can modify 'alphas' for different regularization strengths
vgg_model_ridgecv = RidgeCV(alphas=[0.01, 0.1, 1, 10, 100], cv=5)
vgg_model_ridgecv.fit(X_train_vgg_scaled, y_train)
vgg_pred_ridgecv = vgg_model_ridgecv.predict(X_test_vgg_scaled)
mae_ridgecv = mean_absolute_error(y_test, vgg_pred_ridgecv)
print("VGG16 Features RidgeCV Best Alpha:", vgg_model_ridgecv.alpha_)
print("VGG16 Features RidgeCV MAE:", mae_ridgecv)

# 7. Lasso Regression with cross-validated alpha
# Adjust 'alphas' and 'max_iter' as needed for your dataset
vgg_model_lassocv = LassoCV(alphas=[0.001, 0.01, 0.1, 1, 10], cv=5, max_iter=5000)
vgg_model_lassocv.fit(X_train_vgg_scaled, y_train)
vgg_pred_lassocv = vgg_model_lassocv.predict(X_test_vgg_scaled)
mae_lassocv = mean_absolute_error(y_test, vgg_pred_lassocv)
print("VGG16 Features LassoCV Best Alpha:", vgg_model_lassocv.alpha_)
print("VGG16 Features LassoCV MAE:", mae_lassocv)


Extracting VGG16 features for training set...
Extracting VGG16 features for test set...
VGG16 Features OLS MAE: 19.56992826461792
VGG16 Features RidgeCV Best Alpha: 100.0
VGG16 Features RidgeCV MAE: 16.581603384017946
VGG16 Features LassoCV Best Alpha: 1.0
VGG16 Features LassoCV MAE: 13.052687740325927
